# Semantic Segmentation with U-Net

## Introduction

### What is Semantic Segmentation?
Semantic segmentation is a computer vision task that assigns a class label to each pixel in an image, effectively dividing the image into regions that correspond to different objects or categories.

For example:

* In a street scene image, semantic segmentation would label pixels as road, car, pedestrian, building, sky, etc.

* The result is a dense prediction map where each pixel belongs to a class, but not to a specific object instance (unlike instance segmentation, which distinguishes between different objects of the same class).

In short: semantic segmentation helps machines **"understand"** the contents of an image at the pixel level.


The goal of this task is to perform semantic segmentation on the [Pascal VOC 2012](http://host.robots.ox.ac.uk/pascal/VOC/voc2012/) dataset using the [U-Net](https://arxiv.org/abs/1505.04597) architecture. The dataset consists of natural images with pixel-level annotations for 20 object classes plus background. However, in our task, we only select and use 5 (object) sub-classes from them.

U-Net is a convolutional network architecture originally designed for biomedical image segmentation that has proven highly effective for general segmentation tasks. We will use the U-Net architecture to perform semantic segmentation on the Pascal VOC 2012 dataset. Meanwhile, a pretrained ResNet34 model is used as the encoder of the U-Net.

A typical deep learning segmentation pipeline includes:

1. Data preparation and preprocessing
2. Neural network definition
3. Loss function definition
4. Training
5. Evaluation using appropriate metrics

Your task is to implement the partially completed code, which contains each step of the pipeline.

To better prepare for this test, you are recommended to read the U-Net paper [1] which illustrates the architecture with its characteristic contracting and expanding paths.

[1] Ronneberger, O., Fischer, P., & Brox, T. (2015). "U-Net: Convolutional Networks for Biomedical Image Segmentation." In Medical Image Computing and Computer-Assisted Intervention (MICCAI).

### Evaluation Metrics
mIoU (mean Intersection over Union) is the standard evaluation metric for semantic segmentation tasks. It measures the average overlap between the predicted segmentation and the ground truth across all classes.

For each class \( c \), the **Intersection over Union (IoU)** is defined as:

$$
\text{IoU}_c = \frac{\text{True Positive}_c}{\text{True Positive}_c + \text{False Positive}_c + \text{False Negative}_c}
$$

Where:
- **True Positive (TP)**: pixels correctly predicted as class \( c \)  
- **False Positive (FP)**: pixels incorrectly predicted as class \( c \)  
- **False Negative (FN)**: pixels of class \( c \) predicted as something else

This formula measures how well the predicted area matches the actual area.

After calculating IoU for each class, **mean IoU (mIoU)** is the **average over all \( C \) classes**:

$$
\text{mIoU} = \frac{1}{C} \sum_{c=1}^{C} \text{IoU}_c
$$

## Submission
Please refer to the ```example_submission.xlsx``` to report your model with hyperparameters that achieved the best performance (measured by mIoU), as well as a full list of the models you have implemented. Your ```.ipynb``` should also be submitted.

Checklist
* ```submission_YourName.xlsx```
* ```your_model_weights.pth```
* ```IOAI2025 Computer Vision Test.ipynb```

> The total points of this test is **100**.

---

## Before Starting
Before your test begins, we offer a comprehensive runnable example to help you become familiar with standard deep learning techniques for semantic segmentation tasks.

### Install necessary packages

In [ ]:
!pip3 install torch torchvision matplotlib

#### Import necessary packages

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

### Download and prepare the dataset
We'll use a simplified version of the Pascal VOC dataset for this example.

In [ ]:
# Download the Pascal VOC dataset
# It takes around 3 minute to complete...
voc_train_original = torchvision.datasets.VOCSegmentation(root='./data', year='2012', image_set='train', download=True, transform=None, target_transform=None)
voc_val_original = torchvision.datasets.VOCSegmentation(root='./data', year='2012', image_set='val', download=True, transform=None, target_transform=None)

In [ ]:
voc_train_original[0][0]

In [ ]:
# Define the 5 selected classes from VOC
SELECTED_VOC_CLASSES = [
    15,  # person
    7,   # car
    12,  # dog
    2,   # bicycle
    9,   # chair
]

# Define the colormap for our 5 selected classes
SELECTED_VOC_COLORMAP = [
    [0, 0, 0],         # background (class 0)
    [192, 128, 128],   # person (class 1)
    [128, 128, 128],   # car (class 2)
    [64, 0, 128],      # dog (class 3)
    [0, 128, 0],     # bicycle
    [192, 0, 0],     # chair
]

# Class mapping: original VOC class index -> new 6-class index (background + 5 selected)
VOC_TO_SELECTED_CLASS_MAPPING = {
    0: 0,   # background -> background
    15: 1,  # person -> class 1
    7: 2,   # car -> class 2
    12: 3,  # dog -> class 3
    2: 4,   # bicycle -> class 4
    9: 5,   # chair -> class 5
}

In [ ]:
# Define transformations
class VOCTransform:
    def __init__(self, target_size=(256, 256)):
        self.target_size = target_size
        self.image_transform = transforms.Compose([
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize(target_size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        image = self.image_transform(image)
        target = self.target_transform(target).squeeze().long()

        # Map VOC classes to selected classes
        target_ = torch.zeros_like(target)
        for voc_class, selected_class in VOC_TO_SELECTED_CLASS_MAPPING.items():
            target_[target == voc_class] = selected_class

        return image, target_


# Create datasets and dataloaders
transform = VOCTransform(target_size=(256, 256))


# Create a filtered dataset class that only includes images with our selected classes
class VOCSegmentationDataset(Dataset):
    # this is a modified version of the VOCSegmentationDataset class
    # it only includes images with our selected classes
    def __init__(self, voc_dataset, selected_classes, transform=None):
        self.voc_dataset = voc_dataset
        self.selected_classes = selected_classes
        self.transform = transform
        self.valid_indices = self._filter_dataset()

    def _filter_dataset(self):
        valid_indices = []
        print("Filtering dataset to include only images with selected classes...")
        for idx in tqdm(range(len(self.voc_dataset))):
            _, target = self.voc_dataset[idx]
            target_array = np.array(target)

            # Check if any of our selected classes are in this image
            contains_selected_class = False
            for class_id in self.selected_classes:
                if np.any(target_array == class_id):
                    contains_selected_class = True
                    break

            if contains_selected_class:
                valid_indices.append(idx)

        print(f"Filtered dataset: {len(valid_indices)} out of {len(self.voc_dataset)} images contain selected classes")
        return valid_indices

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        original_idx = self.valid_indices[idx]
        image, target = self.voc_dataset[original_idx]

        if self.transform:
            image, target = self.transform(image, target)

        return image, target

# Create the filtered datasets
train_dataset = VOCSegmentationDataset(
    voc_train_original,
    selected_classes=SELECTED_VOC_CLASSES,  # Use our 5 selected classes
    transform=transform
)

val_dataset = VOCSegmentationDataset(
    voc_val_original,
    selected_classes=SELECTED_VOC_CLASSES,
    transform=transform
)

# Create dataloaders for the filtered datasets
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# Print dataset sizes
print(f"Original training dataset size: {len(voc_train_original)}")
print(f"Filtered training dataset size: {len(train_dataset)}")
print(f"Original validation dataset size: {len(voc_val_original)}")
print(f"Filtered validation dataset size: {len(val_dataset)}")

Let's visualize some examples from the dataset.

In [ ]:
# Function to convert segmentation mask to RGB image
def decode_segmap(mask):
    r = np.zeros_like(mask).astype(np.uint8)
    g = np.zeros_like(mask).astype(np.uint8)
    b = np.zeros_like(mask).astype(np.uint8)

    for l in range(0, len(SELECTED_VOC_COLORMAP)):
        idx = mask == l
        r[idx] = SELECTED_VOC_COLORMAP[l][0]
        g[idx] = SELECTED_VOC_COLORMAP[l][1]
        b[idx] = SELECTED_VOC_COLORMAP[l][2]

    rgb = np.stack([r, g, b], axis=2)
    return rgb

# Function to display image and segmentation mask
def show_sample(image, mask):
    # Denormalize image
    image = image.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image, 0, 1)

    # Convert mask to RGB
    mask = mask.cpu().numpy()
    mask_rgb = decode_segmap(mask)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(image)
    ax1.set_title('Image')
    ax1.axis('off')

    ax2.imshow(mask_rgb)
    ax2.set_title('Segmentation Mask')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

# Display a few samples
for i in range(3):
    image, mask = train_dataset[i]
    show_sample(image, mask)

### Define a simple segmentation model
First, let's define a simple convolutional network for segmentation.

In [ ]:
class SimpleSegmentationModel(nn.Module):
    def __init__(self, num_classes=6):
        super(SimpleSegmentationModel, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
model = SimpleSegmentationModel().to(device)

In [ ]:
# To save the model
torch.save(model.state_dict(), f'./model.pth')
# To load the model
model.load_state_dict(torch.load(f'./model.pth'))

### Define a loss function

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

### Training loop

In [ ]:
def calculate_iou(pred, target, num_classes=6):
    ious = []
    pred = pred.view(-1)
    target = target.view(-1)

    # Create mask to ignore pixels with label 255
    valid_pixels = target != 255
    pred = pred[valid_pixels]
    target = target[valid_pixels]

    # Ignore IoU for background class
    for cls in range(1, num_classes):  # Exclude background
        pred_inds = pred == cls
        target_inds = target == cls

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            ious.append(float('nan'))  # If there is no ground truth, do not include in evaluation
        else:
            ious.append(intersection / union)

    return torch.tensor(ious).nanmean()

In [ ]:
# Training function
def train_model(model, train_loader, criterion, optimizer, num_epochs=2):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(tqdm(train_loader)):

            inputs = inputs.to(device)
            labels = labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)

            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Print statistics
            running_loss += loss.item()
            if i % 50 == 49:
                print(f"Epoch: {epoch+1}, Batch: {i+1}, Loss: {running_loss/50:.4f}")
                running_loss = 0.0
# Validation function
def validate_model(model, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    num_batches = 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Calculate IoU
            preds = torch.argmax(outputs, dim=1)
            iou = calculate_iou(preds, labels)

            val_loss += loss.item()
            val_iou += iou.item()
            num_batches += 1

    avg_loss = val_loss / num_batches
    avg_iou = val_iou / num_batches

    print(f"Validation - Loss: {avg_loss:.4f}, Mean IoU: {avg_iou:.4f}")
    model.train()
    return avg_loss, avg_iou

In [ ]:
# import torch.cuda as cuda
# cuda.empty_cache()

In [ ]:
# Train the model
train_model(model, train_loader, criterion, optimizer, num_epochs=5)

In [ ]:
validate_model(model, val_loader, criterion)

### Visualize results

In [ ]:
# Function to visualize predictions
def visualize_predictions(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))

    for i in range(num_samples):
        # Get a sample
        image, mask = dataset[i]

        # Make prediction
        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(device)
            output = model(input_tensor)
            pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()

        # Denormalize image
        image = image.cpu().numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = std * image + mean
        image = np.clip(image, 0, 1)

        # Convert masks to RGB
        mask = mask.cpu().numpy()
        mask_rgb = decode_segmap(mask)
        pred_rgb = decode_segmap(pred)

        # Display
        axes[i, 0].imshow(image)
        axes[i, 0].set_title('Image')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask_rgb)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(pred_rgb)
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize some predictions
visualize_predictions(model, val_dataset, num_samples=3)

## Q1 Implementing U-Net for Semantic Segmentation
The simple model above achieves limited performance. In this section, we will improve the performance by implementing the U-Net architecture, which is specifically designed for segmentation tasks. You are required to fill missing parts in the three following parts in Q1:
1. Data augmentation
2. U-Net architecture implementation
3. Loss function

### Q1.1 Implement data augmentation (5 pts)
Data augmentation is crucial for improving model generalization. You are required to fill **2 lines** using PyTorch pre-defined functions:
1. Random horizontal flip with probability 0.5
2. Random rotation between -10 and 10 degrees

In [ ]:
class VOCAugmentedTransform:
    def __init__(self, target_size=(256, 256)):
        self.target_size = target_size
        self.image_transform = transforms.Compose([
            transforms.Resize(target_size),
            ###YOUR CODE HERE###
            # Q1.1 line 1

            # Q1.1 line 2

            ###END OF YOUR CODE###
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize(target_size, interpolation=transforms.InterpolationMode.NEAREST),
            # The same augmentations should be applied to the target
            ###YOUR CODE HERE###
            # Q1.1 line 1 (same as above)

            # Q1.1 line 2 (same as above)

            ###END OF YOUR CODE###
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        # Apply the same random seed for both transforms
        seed = np.random.randint(2147483647)
        torch.manual_seed(seed)

        image = self.image_transform(image)
        target = self.target_transform(target).squeeze().long()

        target_ = torch.zeros_like(target)
        for voc_class, selected_class in VOC_TO_SELECTED_CLASS_MAPPING.items():
            target_[target == voc_class] = selected_class
        return image, target_

# Create datasets with augmented transformations
augmented_transform = VOCAugmentedTransform(target_size=(256, 256))
train_dataset_augmented = VOCSegmentationDataset(
    voc_train_original,
    selected_classes=SELECTED_VOC_CLASSES,
    transform=augmented_transform)

train_loader_augmented = DataLoader(train_dataset_augmented, batch_size=8, shuffle=True, num_workers=2)

### Q1.2 Implement U-Net architecture (15 pts)
The U-Net architecture consists of a contracting path (encoder) and an expansive path (decoder) with skip connections between them. You are required to fill **7 lines** in the U-Net implementation:
1. Implement the first convolutional block in the encoder
2. Implement the skip connection that concatenates encoder features with decoder features
3. Implement the final convolutional layer that maps to the number of classes

In [ ]:
class DoubleConv(nn.Module):
    """(Conv -> BN -> ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=6):
        super(UNet, self).__init__()
        # Encoder (downsampling path)
        # Q1.2 line 1: Implement the first convolutional block
        self.inc = ###YOUR CODE HERE###
        self.down1 = ###YOUR CODE HERE###
        self.down2 = ###YOUR CODE HERE###
        self.down3 = ###YOUR CODE HERE###
        self.down4 = ###YOUR CODE HERE###


        # Decoder (upsampling path)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)  # 1024 because of concatenation

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)  # 512 because of concatenation

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)  # 256 because of concatenation

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)  # 128 because of concatenation

        # Q1.2 line 3: Final convolutional layer
        self.outc = ###YOUR CODE HERE###


    def forward(self, x):
        # Encoder path
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder path with skip connections
        x = self.up1(x5)
        # Q1.2 line 2: Implement skip connection (concatenate x with x4)
        x = ###YOUR CODE HERE###
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)

        x = self.outc(x)
        return x

In [ ]:
# Initialize the U-Net model
unet = UNet(n_channels=3, n_classes=6).to(device)

### Q1.3 Implement an appropriate loss function (5 pts)
For semantic segmentation, we need a loss function that handles class imbalance effectively. You are required to fill **1 line** using a PyTorch pre-defined function:
1. Implement a weighted cross-entropy loss that gives more importance to less frequent classes

In [ ]:
# Calculate class weights based on frequency in the training set
def calculate_class_weights(dataloader, num_classes=6):
    class_counts = torch.zeros(num_classes)
    for _, target in tqdm(dataloader, desc="Calculating class weights"):
        for c in range(num_classes):
            class_counts[c] += (target == c).sum().item()

    # Normalize to get weights (less frequent classes get higher weights)
    class_weights = 1.0 / (class_counts + 1e-10)  # Add small epsilon to avoid division by zero
    class_weights = class_weights / class_weights.sum() * num_classes
    return class_weights

# Calculate class weights
class_weights = calculate_class_weights(train_loader)
class_weights = class_weights.to(device)

# Q1.3 line 1: Define weighted cross-entropy loss
criterion_unet = nn.CrossEntropyLoss(
    ###YOUR CODE HERE###

    ignore_index=255
)

# Define optimizer
optimizer_unet = optim.AdamW(unet.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
train_model(unet, train_loader_augmented, criterion, optimizer_unet, num_epochs=5)

In [ ]:
validate_model(unet, val_loader, criterion_unet)

### Q1.4 Add a pre-trained ResNet backbone to the U-Net architecture (10 pts)
However, training the U-Net from scratch is time-consuming. In this question, you are required to add a pre-trained ResNet backbone to the U-Net architecture and fine-tune the model on the Pascal VOC dataset. You are required to fill **1 line** using PyTorch pre-defined functions.

<!-- ![alt text](resnet34.png "ResNet34") -->

In [ ]:

# Import pre-trained models
from torchvision.models import resnet34, ResNet34_Weights

# Define U-Net with pre-trained ResNet backbone
class UNetWithResNetBackbone(nn.Module):
    def __init__(self, n_classes=6):
        super(UNetWithResNetBackbone, self).__init__()

        # Load pre-trained ResNet34 as encoder
        resnet = resnet34(weights=ResNet34_Weights.DEFAULT)

        # Encoder layers
        # We use the first conv layer of ResNet as the first layer of the encoder
        # This line require you to creates a sequential module containing the first convolutional layer (7x7 with stride 2), followed by batch normalization and ReLU activation from the pre-trained ResNet34 model. You need to use the layers from the resnet model.
        self.encoder1 = ###YOUR CODE HERE###  # 64 channels
        self.pool1 = resnet.maxpool
        self.encoder2 = resnet.layer1  # 64 channels
        self.encoder3 = resnet.layer2  # 128 channels
        self.encoder4 = resnet.layer3  # 256 channels
        self.encoder5 = resnet.layer4  # 512 channels

        # Decoder layers
        self.upconv5 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder5 = DoubleConv(512, 256)

        self.upconv4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(256, 128)

        self.upconv3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(128, 64)

        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(96, 32)  # 96 = 64 + 32

        self.upconv1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(16, 16)

        self.conv_last = nn.Conv2d(16, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)  # 64 channels, 1/2 resolution
        e2 = self.encoder2(self.pool1(e1))  # 64 channels, 1/4 resolution
        e3 = self.encoder3(e2)  # 128 channels, 1/8 resolution
        e4 = self.encoder4(e3)  # 256 channels, 1/16 resolution
        e5 = self.encoder5(e4)  # 512 channels, 1/32 resolution

        # Decoder with skip connections
        d5 = self.upconv5(e5)  # 256 channels, 1/16 resolution
        d5 = torch.cat([d5, e4], dim=1)  # 512 channels (256 + 256)
        d5 = self.decoder5(d5)  # 256 channels

        d4 = self.upconv4(d5)  # 128 channels, 1/8 resolution
        d4 = torch.cat([d4, e3], dim=1)  # 256 channels (128 + 128)
        d4 = self.decoder4(d4)  # 128 channels

        d3 = self.upconv3(d4)  # 64 channels, 1/4 resolution
        d3 = torch.cat([d3, e2], dim=1)  # 128 channels (64 + 64)
        d3 = self.decoder3(d3)  # 64 channels

        d2 = self.upconv2(d3)  # 32 channels, 1/2 resolution
        d2 = torch.cat([d2, e1], dim=1)  # 96 channels (32 + 64)
        d2 = self.decoder2(d2)  # 32 channels

        d1 = self.upconv1(d2)  # 16 channels, original resolution
        d1 = self.decoder1(d1)  # 16 channels

        out = self.conv_last(d1)  # n_classes channels
        return out

# Initialize the model
unet_resnet = UNetWithResNetBackbone(n_classes=6).to(device)

# Define optimizer with different learning rates for pre-trained and new layers
encoder_params = list(unet_resnet.encoder1.parameters()) + \
                list(unet_resnet.encoder2.parameters()) + \
                list(unet_resnet.encoder3.parameters()) + \
                list(unet_resnet.encoder4.parameters()) + \
                list(unet_resnet.encoder5.parameters())

decoder_params = list(unet_resnet.upconv5.parameters()) + \
                list(unet_resnet.decoder5.parameters()) + \
                list(unet_resnet.upconv4.parameters()) + \
                list(unet_resnet.decoder4.parameters()) + \
                list(unet_resnet.upconv3.parameters()) + \
                list(unet_resnet.decoder3.parameters()) + \
                list(unet_resnet.upconv2.parameters()) + \
                list(unet_resnet.decoder2.parameters()) + \
                list(unet_resnet.upconv1.parameters()) + \
                list(unet_resnet.decoder1.parameters()) + \
                list(unet_resnet.conv_last.parameters())

optimizer_transfer = optim.Adam([
    {'params': encoder_params, 'lr': 0.0001},  # Lower learning rate for pre-trained layers
    {'params': decoder_params, 'lr': 0.001}    # Higher learning rate for new layers
])


In [ ]:
train_model(unet_resnet, train_loader_augmented, criterion_unet, optimizer_transfer, num_epochs=5)

In [ ]:
validate_model(unet_resnet, val_loader, criterion_unet)

### Q1.4 Bonus Part (10 pts)
The expected performance is around 35% mIoU if you use the same settings. However, you could try different settings to improve the mIoU. The marking criteria is based on the final mIoU. Higher mIoU is better.

Note: remember to evaluate and make a balance between the training time and the performance.

In [ ]:
train_model(unet_resnet, train_loader_augmented, criterion_unet, optimizer_transfer, num_epochs=50)

## Q2 Further improving the segmentation model
We have achieved better performance using the U-Net architecture. In Q2, you are required to further improve the performance of the segmentation model considering the following three possible directions:
1. Deeper U-Net with residual connections
2. Alternative loss functions

### Q2.1 Deeper U-Net with residual connections (5 pts)
The standard U-Net can be improved by adding residual connections within each block, similar to ResNet. Implement a deeper U-Net with residual connections to improve performance. You are required to fill **1 line** using PyTorch pre-defined functions:
1. Implement skip connection

References:
1. https://arxiv.org/abs/1512.03385 (ResNet paper)
2. https://arxiv.org/abs/1608.04117 (ResUNet)

In [ ]:
# Implement a ResidualBlock for U-Net
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Skip connection
        self.skip = nn.Sequential()
        if in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # Q2.1 line 1: Implement skip connection
        ####YOUR CODE HERE###
        out =
        ###END OF YOUR CODE###
        out = self.relu(out)
        return out

# Implement ResUNet
class ResUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=6):
        super(ResUNet, self).__init__()

        # Encoder
        self.inc = ResidualBlock(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(512, 1024))

        # Decoder
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = ResidualBlock(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = ResidualBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = ResidualBlock(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = ResidualBlock(128, 64)

        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder with skip connections
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)

        x = self.outc(x)
        return x

# Initialize the ResUNet model
resunet = ResUNet(n_channels=3, n_classes=6).to(device)

### Q2.2 Alternative loss functions (5 pts)
Implement alternative loss functions that are better suited for segmentation tasks, such as Focal loss. You are required to fill **several lines** using PyTorch pre-defined functions:|
1. Implement focal loss

References:
1. https://arxiv.org/abs/1708.02002 (Focal Loss)

In [ ]:
# Implement Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, weight=None, ignore_index=255):
        super(FocalLoss, self).__init__()
        self.alpha = alpha # alpha is the balancing parameter
        self.gamma = gamma # gamma is the focusing parameter
        self.weight = weight # weight is the weight of each class
        self.ignore_index = ignore_index # ignore_index is the index of the ignored class
        self.ce_fn = nn.CrossEntropyLoss(weight=weight, ignore_index=ignore_index, reduction='none')

    def forward(self, inputs, targets):
        # Q2.2 Implement focal loss
        ####YOUR CODE HERE###

        ###END OF YOUR CODE###
        return focal_loss.mean()

focal_loss = FocalLoss(weight=class_weights, gamma=2, ignore_index=255)

Question (5 pts): What is the alpha and gamma doing here in the focal loss?

Answer: ###

### Train and evaluate the improved model

In [ ]:
# Train ResUNet with advanced augmentations and combined loss
optimizer_resunet = optim.AdamW(resunet.parameters(), lr=1e-3, weight_decay=1e-4)
train_model(resunet, train_loader, criterion_unet, optimizer_resunet, num_epochs=5)

In [ ]:
validate_model(resunet, val_loader, criterion_unet)

In [ ]:
# Visualize final results
visualize_predictions(resunet, val_dataset, num_samples=5)

## Q3 Get a deeper understanding of semantic segmentation task





In this question, you will explore how to use transfer learning to improve semantic segmentation performance. Specifically, you will use a pre-trained backbone (e.g., ResNet) as the encoder part of your U-Net architecture.

Transfer Learning for Semantic Segmentation



1. Implement a U-Net with a pre-trained ResNet backbone
2. Fine-tune the model on the Pascal VOC dataset
3. Compare the performance with your previous models

### Task 1: Analyze class distribution to identify long-tailed distribution (5 pts)
In this task, you are required to analyze the class distribution to identify long-tailed distribution. You can freely choose the method to visualize the class distribution.


### Task 2: Hyperparameter tuning (10 pts)
In this task, you are required to tune the hyperparameters for your previous models. You may want to analyze the performance of your models with different hyperparameters, for example,
* learning rates
* batch sizes
* optimizers
* loss functions
* data augmentations

### Task 3: Implement a U-Net with other pre-trained models and fine-tune it (15 pts)
In this task, you are required to implement a U-Net with other pre-trained ResNet backbone. You will explore how to use transfer learning to improve semantic segmentation performance.



### Task 4: Compare the different architectures (10 pts)
U-Net is a traditional CNN-based architecture. There are possible other architectures that are better suited for semantic segmentation tasks, like transformer-based architectures. You may choose one or more of the possible architectures and compare their performance with your previous models.